# Détection, Classification et Analyse des Fissures dans les Structures de Bâtiments

## 📌 Contexte général

Les fissures dans les structures de bâtiments constituent un indicateur critique de dégradation structurelle.  
Une détection tardive ou imprécise peut entraîner des risques importants pour la sécurité des occupants ainsi que des coûts élevés de maintenance et de réparation.

Ce projet vise à développer **un système d’intelligence artificielle complet**, capable de :

- détecter automatiquement les fissures à partir d’images,
- classifier les fissures selon leur typologie,
- centraliser et analyser les résultats dans un environnement de type *data warehouse* (Snowflake).

## 🎯 Objectifs du projet

Les objectifs principaux sont :

1. **Détection automatique des fissures** sur des images de structures de bâtiments  
2. **Classification des fissures** en 5 catégories :
   - corrosion  
   - diagonal  
   - horizontal  
   - splitting  
   - vertical  
3. **Mise en place d’un pipeline d’inférence reproductible**  
4. **Stockage, historisation et analyse des résultats dans Snowflake**  

---

## Approche méthodologique

Le projet repose sur le framework **YOLOv8 (Ultralytics)** et adopte une stratégie en deux étapes basée sur le *transfer learning*.

### 🔹 Étape 1 — Pré-entraînement (Détection générique)

- Dataset contenant une seule classe (`crack`)
- Objectif principal :
  - apprendre au modèle à **localiser correctement les fissures**,
  - acquérir des représentations visuelles robustes (textures, contours, discontinuités).

### 🔹 Étape 2 — Fine-tuning (Classification multi-classes)

- Dataset plus restreint avec **5 classes de fissures**
- Objectifs :
  - spécialiser le modèle pour la **classification fine**,
  - améliorer la discrimination entre différents types de fissures.
- Technique utilisée :
  - gel partiel des couches du backbone afin de limiter l’overfitting.

### Étape 3 — Envoi des données vers Snowflake

Cette étape consiste à effectuer l’inférence sur les images avec le modèle fine-tuné, puis à stocker les résultats dans Snowflake pour historisation et analyse.

---

## 🗂️ Description des datasets

### 📁 Dataset 1 — Détection générique des fissures

- Environ **1200 images**
- Nombre de classes : **1**
  - `crack`
- Format des annotations : **YOLO**
- Utilisation :
  - pré-entraînement du modèle
  - apprentissage de la localisation des fissures

---

### 📁 Dataset 2 — Classification des fissures

- Environ **120 images**
- Nombre de classes : **5**
  - corrosion
  - diagonal
  - horizontal
  - splitting
  - vertical
- Format des annotations : **YOLO**
- Utilisation :
  - fine-tuning du modèle
  - spécialisation pour la classification multi-classes

---

## 🏛️ Architecture globale du système

L’architecture globale du système suit un pipeline **end-to-end**, depuis l’acquisition des images jusqu’à l’analyse des résultats.


📓 STRUCTURE DU NOTEBOOK
01_setup_environment
02_dataset_preparation
03_pretraining_yolo
04_finetuning_yolo
05_inference_pipeline
06_snowflake_setup

1️⃣ SETUP ENVIRONNEMENT
Cellule 1 — Installation

In [ ]:
!pip install ultralytics snowflake-connector-python pandas opencv-python matplotlib

Cellule 2 — Imports

In [ ]:
from ultralytics import YOLO
import pandas as pd
import cv2
import json
from pathlib import Path

2️⃣ PRÉPARATION DES DATASETS
Cellule — Vérification rapide labels

### Vérification des labels YOLO pour le dataset de test

Ce code permet de **vérifier l’intégrité des fichiers de labels YOLO** dans le dossier de test.  
Chaque fichier `.txt` contient les annotations pour une image, avec une ligne par objet :

- `class_id` : doit être compris entre 0 et 4 (pour 5 classes de fissures)
- `x_center`, `y_center`, `width`, `height` : coordonnées normalisées entre 0 et 1


In [ ]:
import os

# Définition de la fonction pour vérifier les labels
def check_labels(label_path):
    with open(label_path) as f:
        for line in f:
            values = list(map(float, line.split()))
            assert 0 <= values[0] < 5          # Vérifie la classe
            assert all(0 <= v <= 1 for v in values[1:])  # Vérifie les coordonnées normalisées

# Dossier contenant les fichiers .txt
labels_dir = ""

# Parcours des fichiers et vérification
for label_file in os.listdir(labels_dir):
    if label_file.endswith(".txt"):
        label_path = os.path.join(labels_dir, label_file)
        try:
            check_labels(label_path)
            print(f"{label_file} : OK")
        except AssertionError:
            print(f"{label_file} : problème dans le fichier !")


In [ ]:
import os

# Définition de la fonction pour vérifier les labels
def check_labels(label_path):
    with open(label_path) as f:
        for line in f:
            values = list(map(float, line.split()))
            assert 0 <= values[0] < 5          # Vérifie la classe
            assert all(0 <= v <= 1 for v in values[1:])  # Vérifie les coordonnées normalisées

# Dossier contenant les fichiers .txt
labels_dir = ""

# Parcours des fichiers et vérification
for label_file in os.listdir(labels_dir):
    if label_file.endswith(".txt"):
        label_path = os.path.join(labels_dir, label_file)
        try:
            check_labels(label_path)
            print(f"{label_file} : OK")
        except AssertionError:
            print(f"{label_file} : problème dans le fichier !")


3️⃣ PRÉ-ENTRAÎNEMENT — DATASET 1 (1 CLASSE)
Cellule — Entraînement

### Entraînement du modèle YOLOv8 pour la détection de fissures

Ce code permet de **pré-entraîner un modèle YOLOv8** (`yolov8n`) sur le dataset de fissures.  
On utilise le framework **Ultralytics YOLOv8** avec un dataset défini dans un fichier YAML.

- **`data.yaml`** : contient les chemins vers les images d’entraînement, de validation et la liste des classes.  
- **`yolov8n.pt`** : modèle pré-entraîné léger (nano) utilisé pour le *transfer learning*.


In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data="",
    epochs=100,
    imgsz=640,
    batch=16,
    project="runs_pretrain",
    name="crack_detector",
    pretrained=True
)


Cellule — Exemple de détection sur le dataset de test

### Détection et visualisation des fissures sur des images de test

Ce code utilise le **modèle YOLOv8 pré-entraîné** pour effectuer de l’inférence sur des images de test et visualiser les résultats.

- Les images avec fissures détectées sont séparées de celles sans détection.
- Les résultats sont affichés en **grille avec Matplotlib**, avec les boîtes autour des fissures détectées.


In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os
import math

# ------------------------------
# Chemins et modèle
# ------------------------------
run_dir = "runs_pretrain/crack_detector2"
model_path = os.path.join(run_dir, "weights", "best.pt")
model = YOLO(model_path)

# Dossier contenant les images à tester
test_images_dir = ""

# Vérification du dossier
if not os.path.exists(test_images_dir):
    raise FileNotFoundError(f"Le dossier {test_images_dir} n'existe pas.")

# Lister les images .jpg et .png
test_images = [os.path.join(test_images_dir, f) 
               for f in os.listdir(test_images_dir) 
               if f.lower().endswith((".jpg", ".png"))]

if len(test_images) == 0:
    raise FileNotFoundError("Aucune image trouvée dans le dossier de test.")

# Limiter à 6 images pour l'exemple
test_images = test_images[:6]

# ------------------------------
# Détection et séparation
# ------------------------------
crack_images = []
non_crack_images = []

for img_path in test_images:
    results = model.predict(img_path, conf=0.25, save=False)
    
    # Vérifier si des boîtes ont été détectées
    if results[0].boxes is None or results[0].boxes.xyxy.shape[0] == 0:
        non_crack_images.append(img_path)
    else:
        crack_images.append((img_path, results[0].plot()))  # Image avec boxes

# ------------------------------
# Affichage en grille
# ------------------------------
def show_images_grid(images, title_prefix, max_cols=3):
    n = len(images)
    cols = min(n, max_cols)
    rows = math.ceil(n / cols)
    plt.figure(figsize=(5*cols, 4*rows))
    
    for i, img_info in enumerate(images):
        plt.subplot(rows, cols, i+1)
        if isinstance(img_info, tuple):
            # Image avec boxes
            img = img_info[1]
            name = os.path.basename(img_info[0])
        else:
            # Image sans boxes
            img = plt.imread(img_info)
            name = os.path.basename(img_info)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"{title_prefix}: {name}")
    plt.tight_layout()
    plt.show()

# 🔹 Affichage cracks
if crack_images:
    show_images_grid(crack_images, "Crack")

# 🔹 Affichage non-cracks
if non_crack_images:
    show_images_grid(non_crack_images, "Non-Crack")


Cellule TEST D’UNE IMAGE EXTERNE — VISUALISATION ET PARAMÈTRES DE DÉTECTION

### Détection sur une image externe

Ce code permet de réaliser la **détection d’une fissure sur une image unique** avec le modèle YOLOv8 fine-tuné.  

- L’image peut provenir d’un dossier externe.
- On affiche **l’image annotée avec les boîtes** et on imprime les détails de chaque détection (classe, confiance et coordonnées).


In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os

# ------------------------------
# Charger le modèle
# ------------------------------
run_dir = "runs_pretrain/crack_detector2"
model_path = os.path.join(run_dir, "weights", "best.pt")
model = YOLO(model_path)

# ------------------------------
# Chemin de l'image externe
# ------------------------------
image_path = "crack_detection_project/Image_detection/test_img7.jpg"  # <- Remplace par ton image

if not os.path.exists(image_path):
    raise FileNotFoundError(f"L'image {image_path} n'existe pas.")

# ------------------------------
# Détection
# ------------------------------
results = model.predict(image_path, conf=0.25, save=False)

# ------------------------------
# Affichage de l'image avec boîtes
# ------------------------------
img_with_boxes = results[0].plot()
plt.figure(figsize=(8,6))
plt.imshow(img_with_boxes)
plt.axis('off')
plt.title(f"Détection : {os.path.basename(image_path)}")
plt.show()

# ------------------------------
# Affichage des résultats détaillés
# ------------------------------
if len(results[0].boxes) == 0:
    print("Aucune détection sur cette image.")
else:
    for i, box in enumerate(results[0].boxes):
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()  # [x1, y1, x2, y2]
        print(f"Détection {i+1} : Classe = {results[0].names[cls_id]}, "
              f"Confiance = {conf:.2f}, Coordonnées = {xyxy}")


Cellule TEST D’UN ENSEMBLE D'IMAGES EXTERNE — VISUALISATION ET PARAMÈTRES DE DÉTECTION

### Détection sur un dossier d'images et affichage en grille

Ce code permet de :  

1. Charger le modèle YOLOv8 fine-tuné.  
2. Parcourir un dossier d’images (`.jpg` ou `.png`).  
3. Effectuer la détection de fissures sur chaque image.  
4. Afficher toutes les images annotées avec les boîtes dans une **grille**, en indiquant le nombre de détections.


In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os
import pandas as pd
import math

# ------------------------------
# Chemins et modèle
# ------------------------------
run_dir = "runs_pretrain/crack_detector2"
model_path = os.path.join(run_dir, "weights", "best.pt")
model = YOLO(model_path)

# Dossier contenant les images à tester
test_images_dir = "crack_detection_project/Image_detection"

# Vérification du dossier
if not os.path.exists(test_images_dir):
    raise FileNotFoundError(f"Le dossier {test_images_dir} n'existe pas.")

# Lister toutes les images
test_images = [os.path.join(test_images_dir, f) 
               for f in os.listdir(test_images_dir) 
               if f.lower().endswith((".jpg", ".png"))]

if len(test_images) == 0:
    raise FileNotFoundError("Aucune image trouvée dans le dossier de test.")

# ------------------------------
# Détection et préparation des images
# ------------------------------
images_with_boxes = []
titles = []

for img_path in test_images:
    results = model.predict(img_path, conf=0.25, save=False)
    img_with_boxes = results[0].plot()
    images_with_boxes.append(img_with_boxes)
    
    # Préparer le titre avec le nombre de détections
    det = results[0].boxes
    if det is not None and len(det) > 0:
        title = f"{os.path.basename(img_path)} — {len(det)} détecté(s)"
    else:
        title = f"{os.path.basename(img_path)} — Aucun objet détecté"
    titles.append(title)

# ------------------------------
# Affichage en grille
# ------------------------------
num_images = len(images_with_boxes)
cols = 3  # nombre de colonnes dans la grille
rows = math.ceil(num_images / cols)

plt.figure(figsize=(cols*5, rows*4))

for i, img in enumerate(images_with_boxes):
    plt.subplot(rows, cols, i+1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(titles[i], fontsize=10)

plt.tight_layout()
plt.show()



4️⃣ FINE-TUNING — DATASET 2 (5 CLASSES)

1. corrosion  
2. diagonal  
3. horizontal  
4. splitting  
5. vertical

---

#### Étape 1 : Chargement du modèle pré-entraîné

Cellule — Chargement

In [6]:
model = YOLO("runs_pretrain/crack_detector2/weights/best.pt")

#### Étape 2 : Gel partiel du backbone

Pour limiter l’overfitting sur un dataset restreint, on **freeze les premières couches** du modèle.  
Ici, on fige les **10 premières couches du backbone** pour que seules les couches supérieures soient entraînables.

- `requires_grad_(False)` : empêche la mise à jour des poids lors du backpropagation.
- Cette technique permet d’accélérer l’entraînement et de préserver les caractéristiques visuelles déjà apprises.


In [ ]:
model.model.model[:10].requires_grad_(False)


Cellule — Fine-tuning
### 🔹 Paramètres principaux de `model.train()`

- **`data`** : chemin vers le dataset YAML (train, val, noms des classes).  
- **`epochs`** : nombre de passages sur tout le dataset (ici 80).  
- **`imgsz`** : taille des images redimensionnées (640×640).  
- **`batch`** : nombre d’images par batch (ici 8).  
- **`lr0`** : learning rate initial (1e-4 pour fine-tuning).  
- **`lrf`** : facteur pour réduire progressivement le learning rate (1e-2).  
- **`project` / `name`** : dossier et nom du run pour sauvegarder poids et résultats.


In [ ]:
model.train(
    data="crack_detection_project/datasets/dataset_crack_5classes/data.yaml",
    epochs=80,
    imgsz=640,
    batch=8,
    lr0=1e-4,
    lrf=1e-2,
    project="runs_finetune",
    name="crack_5classes"
)


### Test d’un ensemble d’images externes — Visualisation et paramètres de détection

Cette cellule permet de **tester le modèle fine-tuné sur de nouvelles images** et d’afficher les résultats dans une **grille** avec le nombre de détections et les classes détectées.

---

**Étapes principales :**

1. **Chargement du modèle fine-tuné**  
   - On utilise le modèle sauvegardé après le fine-tuning sur le dataset 5 classes (`best.pt`).

2. **Chargement du dossier d’images à tester**  
   - Toutes les images `.jpg` et `.png` sont listées automatiquement.  
   - Une exception est levée si le dossier est vide ou inexistant.

3. **Détection pour chaque image**  
   - `model.predict(img_path, conf=0.25, save=False)` :  
     - `conf=0.25` : seuil de confiance pour filtrer les détections faibles.  
     - `save=False` : on ne sauvegarde pas les images automatiquement.  
   - Les images sont ensuite annotées avec les boîtes et stockées pour l’affichage.

4. **Préparation des titres**  
   - Pour chaque image, le titre indique :  
     - le nom du fichier  
     - le nombre d’objets détectés  
     - les classes détectées (`corrosion`, `diagonal`, `horizontal`, `splitting`, `vertical`)  
   - Si aucune détection → titre “Aucun objet détecté”.

5. **Affichage en grille**  
   - Nombre de colonnes : 3  
   - Calcul automatique du nombre de lignes en fonction du nombre d’images  
   - Utilisation de `matplotlib` pour afficher les images annotées avec leurs titres.


In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os
import math

# ------------------------------
# Chemins et modèle (FINETUNE)
# ------------------------------
run_dir = "runs_finetune/crack_5classes2"
model_path = os.path.join(run_dir, "weights", "best.pt")
model = YOLO(model_path)

# Dossier contenant les images à tester
test_images_dir = "crack_detection_project/Image_detection"

# Vérification du dossier
if not os.path.exists(test_images_dir):
    raise FileNotFoundError(f"Le dossier {test_images_dir} n'existe pas.")

# Lister toutes les images
test_images = [os.path.join(test_images_dir, f) 
               for f in os.listdir(test_images_dir) 
               if f.lower().endswith((".jpg", ".png"))]

if len(test_images) == 0:
    raise FileNotFoundError("Aucune image trouvée dans le dossier de test.")

# ------------------------------
# Détection et préparation des images
# ------------------------------
images_with_boxes = []
titles = []

for img_path in test_images:
    results = model.predict(img_path, conf=0.25, save=False)
    img_with_boxes = results[0].plot()
    images_with_boxes.append(img_with_boxes)
    
    # Préparer le titre avec le nombre de détections et les classes
    det = results[0].boxes
    if det is not None and len(det) > 0:
        cls_ids = det.cls.cpu().numpy().astype(int)
        class_names = [model.names[c] for c in cls_ids]
        title = f"{os.path.basename(img_path)} — {len(det)} détecté(s): {', '.join(class_names)}"
    else:
        title = f"{os.path.basename(img_path)} — Aucun objet détecté"
    titles.append(title)

# ------------------------------
# 3️⃣ Affichage en grille
# ------------------------------
num_images = len(images_with_boxes)
cols = 3  # nombre de colonnes dans la grille
rows = math.ceil(num_images / cols)

plt.figure(figsize=(cols*5, rows*4))

for i, img in enumerate(images_with_boxes):
    plt.subplot(rows, cols, i+1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(titles[i], fontsize=10)

plt.tight_layout()
plt.show()



### 5️⃣ Pipeline d’inférence — Fonction d’inférence pour une image

Cette cellule définit une **fonction d’inférence** qui permet de détecter les fissures sur une image à partir du modèle **fine-tuné** et de retourner les résultats sous forme de **DataFrame**.

---

**Étapes principales :**

1. **Chargement du modèle fine-tuné**  
   - Le modèle `best.pt` issu du fine-tuning sur le dataset 5 classes est chargé.

2. **Définition de la fonction `run_inference`**  
   - Entrée : chemin de l’image et modèle YOLO  
   - Sortie : liste de dictionnaires avec les informations des détections :  
     - `class_id` : ID de la classe détectée  
     - `confidence` : confiance de la détection  
     - `x_center`, `y_center` : coordonnées du centre normalisées  
     - `width`, `height` : largeur et hauteur normalisées  

3. **Exemple d’inférence sur une image**  
   - L’image `test_img1.jpg` est passée dans la fonction.  
   - Les résultats sont convertis en `pandas.DataFrame` pour un affichage clair et tabulaire.


In [ ]:
# Importer YOLO et pandas si ce n'est pas déjà fait
from ultralytics import YOLO
import pandas as pd

# Charger le modèle fine-tuné
model_path = "crack_detection_project/notebook/runs_finetune/crack_5classes3/weights/best.pt"
model = YOLO(model_path)

# Définir la fonction d'inférence
def run_inference(image_path, model):
    results = model(image_path)[0]
    detections = []

    for box in results.boxes:
        detections.append({
            "class_id": int(box.cls),
            "confidence": float(box.conf),
            "x_center": float(box.xywhn[0][0]),
            "y_center": float(box.xywhn[0][1]),
            "width": float(box.xywhn[0][2]),
            "height": float(box.xywhn[0][3])
        })
    return detections

# Chemin de l'image à tester
image_path = "crack_detection_project/Image_detection/test_img1.jpg"

# Faire l'inférence
detections = run_inference(image_path, model)

# Afficher les résultats sous forme de DataFrame
df = pd.DataFrame(detections)
print(df)



### 6️⃣ Détection pour une image unique et envoi vers Snowflake

Cette cellule combine **inférence YOLOv8** et **envoi automatique des résultats** vers une table Snowflake.

---

**Étapes principales :**

1. **Chargement du modèle fine-tuné**  
   - Le modèle `best.pt` issu du fine-tuning sur le dataset 5 classes est chargé.

2. **Définition de la fonction `run_inference`**  
   - Entrée : chemin de l’image et modèle YOLO  
   - Sortie : liste de dictionnaires avec :  
     - `class_id` : ID de la classe détectée  
     - `confidence` : confiance de la détection  
     - `x_center`, `y_center`, `width`, `height` : coordonnées normalisées  
     - `image_width`, `image_height` : dimensions originales de l’image  
     - `preprocess_time`, `inference_time`, `postprocess_time` : temps en millisecondes pour chaque étape

3. **Définition de la fonction `send_to_snowflake`**  
   - Connexion à Snowflake avec les paramètres (`user`, `password`, `account`, `warehouse`, `database`, `schema`)  
   - Boucle sur toutes les détections et insertion dans la table `crack_detections`  
   - Les informations sauvegardées incluent dimensions de l’image, nombre de détections et temps de traitement.  
   - Validation avec `conn.commit()` et fermeture propre de la connexion.

4. **Exemple d’utilisation**  
   - On teste l’image `test_img7.jpg`.  
   - Les résultats sont affichés sous forme de `DataFrame` pour vérification.  
   - Ensuite, les détections sont envoyées automatiquement dans Snowflake.

---

💡 **Remarques pratiques :**  
- La fonction peut être adaptée pour **traiter plusieurs images** dans un dossier.  
- Les temps de traitement permettent de **monitorer la performance** du pipeline d’inférence.  


In [ ]:
# Importer les librairies
from ultralytics import YOLO
import pandas as pd
import snowflake.connector
import os

# Charger le modèle fine-tuné
model_path = "crack_detection_project/notebook/runs_finetune/crack_5classes3/weights/best.pt"
model = YOLO(model_path)

# Fonction d'inférence
def run_inference(image_path, model):
    results = model(image_path)[0]
    detections = []

    # Extraire infos image et timing
    image_height, image_width = results.orig_shape[:2]
    timings = results.speed  # dict avec preprocess, inference, postprocess
    preprocess_time = timings['preprocess'] * 1000  # ms
    inference_time = timings['inference'] * 1000
    postprocess_time = timings['postprocess'] * 1000

    for box in results.boxes:
        detections.append({
            "class_id": int(box.cls),
            "confidence": float(box.conf),
            "x_center": float(box.xywhn[0][0]),
            "y_center": float(box.xywhn[0][1]),
            "width": float(box.xywhn[0][2]),
            "height": float(box.xywhn[0][3]),
            "image_width": image_width,
            "image_height": image_height,
            "preprocess_time": preprocess_time,
            "inference_time": inference_time,
            "postprocess_time": postprocess_time
        })
    return detections

# Fonction pour envoyer les données à Snowflake
def send_to_snowflake(detections, image_name):
    conn = snowflake.connector.connect(
    user="",
    password="",
    account="",
    warehouse="GECKO_WH",
    database="GECKO_CRACK_AI",
    schema="PUBLIC"
    )
    cursor = conn.cursor()

    for det in detections:
        cursor.execute("""
            INSERT INTO crack_detections 
            (image_name, class_id, confidence, x_center, y_center, width, height,
             image_size, detections_count, preprocess_time, inference_time, postprocess_time)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            image_name,
            det["class_id"],
            det["confidence"],
            det["x_center"],
            det["y_center"],
            det["width"],
            det["height"],
            f'{det["image_width"]}x{det["image_height"]}',
            len(detections),
            det["preprocess_time"],
            det["inference_time"],
            det["postprocess_time"]
        ))

    conn.commit()
    cursor.close()
    conn.close()
    print(f"{len(detections)} détection(s) envoyée(s) à Snowflake pour l'image {image_name}.")

# Chemin de l'image à tester
image_path = "crack_detection_project/Image_detection/test_img7.jpg"
image_name = os.path.basename(image_path)  # Extraire le nom du fichier

# Faire l'inférence
detections = run_inference(image_path, model)

# Créer le DataFrame pour visualisation (optionnel)
df = pd.DataFrame(detections)
print(df)

# Envoyer les résultats directement à Snowflake
send_to_snowflake(detections, image_name)


# Détection de fissures pour toutes les images d'un dossier et envoi vers Snowflake
### 🔹 Détection en batch et envoi vers Snowflake

Cette cellule parcourt toutes les images d’un dossier, effectue l’inférence avec le **modèle fine-tuné YOLOv8** et envoie les résultats à Snowflake.  

- Extraction des **boîtes détectées** (`class_id`, `confidence`...) et informations sur l’image.  
- Calcul des **temps de traitement** (`preprocess`, `inference`, `postprocess`).  
- Insertion directe des résultats dans la table `crack_detections`.  
- Affichage optionnel sous forme de `DataFrame` pour vérification.



In [ ]:
# Importer les librairies
from ultralytics import YOLO
import pandas as pd
import snowflake.connector
import os
from glob import glob

# Charger le modèle fine-tuné
model_path = "crack_detection_project/notebook/runs_finetune/crack_5classes3/weights/best.pt"
model = YOLO(model_path)

# Fonction d'inférence pour une image
def run_inference(image_path, model):
    results = model(image_path)[0]
    detections = []

    # Extraire infos image et timing
    image_height, image_width = results.orig_shape[:2]
    timings = results.speed
    preprocess_time = timings['preprocess'] * 1000
    inference_time = timings['inference'] * 1000
    postprocess_time = timings['postprocess'] * 1000

    for box in results.boxes:
        detections.append({
            "class_id": int(box.cls),
            "confidence": float(box.conf),
            "x_center": float(box.xywhn[0][0]),
            "y_center": float(box.xywhn[0][1]),
            "width": float(box.xywhn[0][2]),
            "height": float(box.xywhn[0][3]),
            "image_width": image_width,
            "image_height": image_height,
            "preprocess_time": preprocess_time,
            "inference_time": inference_time,
            "postprocess_time": postprocess_time
        })
    return detections

# Fonction pour envoyer les détections à Snowflake
def send_to_snowflake(detections, image_name):
    if len(detections) == 0:
        print(f"Aucune détection pour l'image {image_name}. Ignorée.")
        return  # Ne rien envoyer si pas de détection

    conn = snowflake.connector.connect(
        user="",
        password="",
        account="",
        warehouse="GECKO_WH",
        database="GECKO_CRACK_AI",
        schema="PUBLIC"
    )
    cursor = conn.cursor()

    for det in detections:
        cursor.execute("""
            INSERT INTO crack_detections 
            (image_name, class_id, confidence, x_center, y_center, width, height,
             image_size, detections_count, preprocess_time, inference_time, postprocess_time)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            image_name,
            det["class_id"],
            det["confidence"],
            det["x_center"],
            det["y_center"],
            det["width"],
            det["height"],
            f'{det["image_width"]}x{det["image_height"]}',
            len(detections),
            det["preprocess_time"],
            det["inference_time"],
            det["postprocess_time"]
        ))

    conn.commit()
    cursor.close()
    conn.close()
    print(f"{len(detections)} détection(s) envoyée(s) à Snowflake pour l'image {image_name}.")

# Traiter toutes les images dans un dossier
image_folder = "crack_detection_project/Image_detection/"
image_paths = glob(os.path.join(image_folder, "*.*"))  # tous les fichiers

for image_path in image_paths:
    image_name = os.path.basename(image_path)
    print(f"Traitement de l'image : {image_name}")

    detections = run_inference(image_path, model)
    df = pd.DataFrame(detections)
    print(df)  # Optionnel pour vérifier les résultats

    send_to_snowflake(detections, image_name)


### 🔹 Création de la table `crack_detections` et visualisation des résultats

La table `crack_detections` permet de **stocker toutes les informations des détections de fissures** pour analyse ultérieure.

```sql
CREATE OR REPLACE TABLE crack_detections (
    image_name STRING,            -- Nom du fichier image
    class_id INT,                 -- ID de la classe de fissure détectée
    confidence FLOAT,             -- Confiance de la détection (0 à 1)
    x_center FLOAT,               -- Coordonnée X du centre de la boîte (normalisée)
    y_center FLOAT,               -- Coordonnée Y du centre de la boîte (normalisée)
    width FLOAT,                  -- Largeur de la boîte (normalisée)
    height FLOAT,                 -- Hauteur de la boîte (normalisée)
    image_size STRING,            -- Dimensions de l'image (ex: 640x480)
    detections_count INT,         -- Nombre total de détections sur l'image
    preprocess_time FLOAT,        -- Temps de prétraitement en ms
    inference_time FLOAT,         -- Temps d'inférence du modèle en ms
    postprocess_time FLOAT,       -- Temps de post-traitement en ms
    detection_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP  -- Date et heure de la détection
);

Les requêtes suivantes permettent d’explorer et analyser les détections stockées dans la table `crack_detections`.

```sql
-- 1️⃣ Voir toutes les détections
SELECT * 
FROM crack_detections;
-- Affiche toutes les lignes de la table pour vérifier l'ensemble des détections.

-- 2️⃣ Compter les détections par classe
SELECT class_id, COUNT(*) AS count
FROM crack_detections
GROUP BY class_id;
-- Montre combien de fissures ont été détectées pour chaque type (classe).

-- 3️⃣ Filtrer les détections avec une confiance > 0.8
SELECT * 
FROM crack_detections
WHERE confidence > 0.8;
-- Permet d’afficher uniquement les détections les plus fiables.

-- 4️⃣ Compter les détections par image
SELECT image_name, COUNT(*) AS count
FROM crack_detections
GROUP BY image_name;
-- Indique combien de fissures ont été détectées pour chaque image.

-- 5️⃣ Voir les détections avec toutes les infos d’image et timing
SELECT image_name, class_id, confidence, x_center, y_center, width, height,
       image_size, detections_count, preprocess_time, inference_time, postprocess_time, detection_time
FROM crack_detections;
-- Affiche toutes les informations pour chaque détection : coordonnées, taille, temps de traitement et date.